# GeoZones mapping

For each dataset in the export CSV that has a `spatial.geom` bbox, find the best-matching
administrative zone from `zones_filtered.geojson`.

**Strategy**: `sjoin(predicate='intersects')` finds candidate pairs, then actual
`shapely.intersection` area gives real IoU scores. Best match per dataset must satisfy:
- `IoU ≥ 0.4`
- `zone_area ≤ bbox_area` (zone must not dwarf the bbox)

In [71]:
import geopandas as gpd
import pandas as pd
import json
from shapely.geometry import shape

DATASET_URL = "https://www.data.gouv.fr/api/1/datasets/r/f868cca6-8da1-4369-a78d-47463f19a9a3"
DATASET_CSV = "./export-dataset.csv"
ZONES_FILE  = "./zones_filtered.geojson"
CRS_GEO     = "EPSG:4326"

print("geopandas", gpd.__version__)

geopandas 1.1.4


In [72]:
import urllib.request, os

if not os.path.exists(DATASET_CSV):
    print(f"Downloading {DATASET_URL} ...")
    urllib.request.urlretrieve(DATASET_URL, DATASET_CSV)
    print(f"Done. ({os.path.getsize(DATASET_CSV)/1024/1024:.1f} MB)")
else:
    print(f"Already exists: {DATASET_CSV}  ({os.path.getsize(DATASET_CSV)/1024/1024:.1f} MB)")

Done. (231.4 MB)


## Load data

In [73]:
# Load zones — keep only the columns we need for matching
zones = gpd.read_file(ZONES_FILE, engine="pyogrio")[["id", "geometry"]].rename(columns={"id": "zone_id"})
zones = zones.set_crs(CRS_GEO)
print(f"Zones: {len(zones)} features")
zones.head(3)

Zones: 38340 features


,zone_id,geometry
0,country:fr,"MULTIPOLYGON (((-52.55642 2.50471, -52.93966 2..."
1,country-group:ue,"MULTIPOLYGON (((-52.55642 2.50471, -52.93966 2..."
2,country-group:world,"MULTIPOLYGON (((-159.20818 -79.49706, -161.127..."


In [74]:
# Load CSV, parse spatial.geom (GeoJSON strings) into shapely geometries
df = pd.read_csv(DATASET_CSV, sep=";", dtype=str)

def parse_geom(raw):
    if not isinstance(raw, str) or not raw.strip():
        return None
    try:
        return shape(json.loads(raw))
    except Exception:
        return None

df["geometry"] = df["spatial.geom"].apply(parse_geom)
datasets = gpd.GeoDataFrame(df, geometry="geometry", crs=CRS_GEO)

has_geom = datasets[datasets.geometry.notna()].copy()
print(f"Total rows: {len(datasets)}  |  with geometry: {len(has_geom)}")
has_geom[["id", "title", "spatial.granularity", "geometry"]].head(3)

Total rows: 126831  |  with geometry: 45431


,id,title,spatial.granularity,geometry
0,6a45ef775417274350b559e3,Secteurs concernés par la migration et l'hiver...,other,"MULTIPOLYGON (((8.23029 50.16764, 3.38409 50.1..."
1,6a45ef775417274350b559e2,Secteurs concernés par un fort risque d'érosio...,other,"MULTIPOLYGON (((8.23029 50.16764, 3.38409 50.1..."
2,6a45ef765417274350b559e1,Secteurs d'allongement des périodes d'interdic...,other,"MULTIPOLYGON (((8.23029 50.16764, 3.38409 50.1..."


## Matching

In [75]:
import shapely
import numpy as np
import time

IOU_THRESHOLD = 0.4

# zones_filtered.geojson is already clean: no @date suffixes, no obsolete regions.
# Multiple historical versions of the same zone share the same clean ID — sjoin uses
# all of them (historical boundaries matter), dedup only happens for the geometry lookup.
zones_geo = zones.copy()
zones_geo["geometry"]  = shapely.make_valid(zones_geo.geometry.values)
zones_geo["zone_area"] = shapely.area(zones_geo.geometry.values)

sample_geo = has_geom[["id", "geometry"]].copy()
sample_geo["bbox_area"] = shapely.area(sample_geo.geometry.values)

t0 = time.perf_counter()

candidates = gpd.sjoin(
    sample_geo,
    zones_geo[["zone_id", "zone_area", "geometry"]],
    how="inner",
    predicate="intersects",
).drop(columns="index_right")

t1 = time.perf_counter()
print(f"sjoin: {t1-t0:.1f}s  — {len(candidates):,} candidate pairs")

# Stage 1: area-ratio pre-filter (lossless)
# max possible IoU = min_area/max_area, so any pair below IOU_THRESHOLD can never pass
lo = np.minimum(candidates["zone_area"].values, candidates["bbox_area"].values)
hi = np.maximum(candidates["zone_area"].values, candidates["bbox_area"].values)
plausible = candidates[(hi > 0) & (lo / hi >= IOU_THRESHOLD)].copy()
print(f"Area pre-filter: {len(candidates):,} → {len(plausible):,} candidates")

# Stage 2: actual IoU on plausible candidates
# One geometry per zone_id (first occurrence) for the intersection lookup
zone_geom_lookup = zones_geo.drop_duplicates("zone_id").set_index("zone_id")["geometry"]
bbox_geoms_arr = shapely.make_valid(plausible.geometry.values)
zone_geoms_arr = shapely.make_valid(zone_geom_lookup.reindex(plausible["zone_id"]).values)

inter_area = shapely.area(shapely.intersection(bbox_geoms_arr, zone_geoms_arr))
union_area = plausible["bbox_area"].values + plausible["zone_area"].values - inter_area
plausible["iou_score"] = np.where(union_area > 0, inter_area / union_area, 0.0)

t2 = time.perf_counter()
print(f"Actual IoU: {t2-t1:.1f}s")

# Stage 3: best match — IoU ≥ threshold and zone fits inside bbox
best = (
    plausible[
        (plausible["iou_score"] >= IOU_THRESHOLD) &
        (plausible["zone_area"] <= plausible["bbox_area"])
    ]
    .sort_values("iou_score", ascending=False)
    .drop_duplicates(subset="id", keep="first")
    [["id", "zone_id", "iou_score"]]
)

t3 = time.perf_counter()
print(f"\nTotal: {t3-t0:.0f}s — {len(best):,} matched / {len(has_geom):,} datasets with geometry")

sjoin: 40.4s  — 132,596,454 candidate pairs


Area pre-filter: 132,596,454 → 174,478 candidates


Actual IoU: 78.0s

Total: 118s — 32,345 matched / 45,431 datasets with geometry


In [76]:
output = best.merge(
    has_geom[["id", "title"]],
    on="id", how="left"
)[["id", "title", "zone_id", "iou_score"]]

assert output["iou_score"].between(0, 1).all(), "IoU out of [0,1]"

OUT_CSV = "./geozones-mapping-output.csv"
output.to_csv(OUT_CSV, index=False)
print(f"Saved {len(output):,} rows → {OUT_CSV}")

Saved 32,345 rows → ./geozones-mapping-output.csv


In [77]:
output["level"] = output["zone_id"].str.extract(r"^([^:]+:[^:]+)")
print(output["level"].value_counts().to_string())

level
fr:departement       12376
fr:region             9581
fr:commune            7486
fr:arrondissement     1136
country-subset:fr      725
fr:epci                630
fr:canton              365
country:fr              56


level
fr:departement       12508
fr:region             8429
fr:commune            7310
fr:arrondissement      837
country-subset:fr      680
fr:epci                435
fr:canton              268
country:fr              56


level
fr:departement       12508
fr:region             8429
fr:commune            7323
fr:arrondissement      837
country-subset:fr      680
fr:epci                405
fr:canton              268
country:fr              56


level
fr:departement       12508
fr:region             8429
fr:commune            7310
fr:arrondissement      837
country-subset:fr      680
fr:epci                435
fr:canton              268
country:fr              56


level
fr:departement       14657
fr:region             8390
fr:commune            7358
fr:epci               1197
country-subset:fr      680
country:fr              56


level
fr:departement       14656
fr:region             8393
fr:commune            7363
fr:epci               1197
country-subset:fr      680
country:fr              56


In [78]:
output.sample(10)

,id,title,zone_id,iou_score,level
8392,684c4c5945e1b1cfad52b6ad,Localisation des Incinérateurs sur Auvergne-Rh...,fr:region:84,0.592769,fr:region
28112,67377f87b561510ce3c254b0,Plan de Prévention des Risques Naturels (PPRN)...,fr:commune:65397,0.471945,fr:commune
25675,697954914ba57932dd986892,"Ports, villes et réseaux de communication au I...",fr:region:32,0.489329,fr:region
7384,69be17d0bc7ea8b6921580dc,ilévia - Vitesse moyenne des bus par inter-arrêts,fr:epci:200093201,0.600661,fr:epci
6442,6734d0d13b82cd64f8c40ac2,Périmètres des communes concernées par l'aléa ...,fr:departement:40,0.604952,fr:departement
796,67cfa1a231bb50147624f7ac,Zonages réglementaires d'un PPRN ou d'un PPRM ...,fr:departement:51,0.688274,fr:departement
15657,6a0511815239fe1f443881cf,DONNEE - Territoire autorité organisatrice de ...,fr:region:44,0.530566,fr:region
641,637d65638ea357e1592908e8,DONNEE - Dallages des orthophotos - Départemen...,fr:departement:51,0.688320,fr:departement
7360,6916a95f6ce819db23a37686,Schéma cyclable 2035 - points durs - Plan de m...,fr:epci:200093201,0.600661,fr:epci
4045,6835e1aab63a9d35e7858956,Zones soustraites à l’inondation (protégées de...,fr:region:11,0.626040,fr:region
